# 🎓 SyllaBot — Checkpoint 1: Embedding Generation Demo
**IS Professional Elective #4 — Generative AI Systems**  
**Group: Sugar Group | A.Y. 2026–2027**

This notebook covers **Deliverable #4** of Checkpoint 1:
- Upload cleaned `.txt` files from the preprocessing step
- Split text into chunks for embedding
- Generate embeddings using HuggingFace `all-MiniLM-L6-v2`
- Visualize and explain the embeddings
- Save embeddings for use in Checkpoint 2


## Step 1 — Install Required Libraries

In [ ]:
!pip install sentence-transformers numpy pandas matplotlib -q

## Step 2 — Upload Your Cleaned `.txt` Files
Upload the `.txt` files saved by the preprocessing notebook (`syllabi_cleaned/` folder).

In [ ]:
from google.colab import files
import os

os.makedirs('syllabi_cleaned', exist_ok=True)

print('📂 Upload your cleaned .txt files...')
uploaded = files.upload()

for filename in uploaded.keys():
    os.rename(filename, f'syllabi_cleaned/{filename}')
    print(f'  ✅ Saved: {filename}')

print(f'\n📁 Total files uploaded: {len(uploaded)}')

## Step 3 — Load All Cleaned Text Files

In [ ]:
documents = {}  # {filename: full_text}

for filename in sorted(os.listdir('syllabi_cleaned')):
    if filename.endswith('.txt'):
        with open(f'syllabi_cleaned/{filename}', 'r', encoding='utf-8') as f:
            text = f.read().strip()
            documents[filename] = text
            print(f'  📄 Loaded: {filename} ({len(text):,} characters)')

print(f'\n✅ Total documents loaded: {len(documents)}')

## Step 4 — Split Text into Chunks
We cannot embed an entire document at once — it would be too long.
We split each document into smaller overlapping chunks of ~500 characters.

**Why overlap?** So that sentences at chunk boundaries are not cut off and lost.

In [ ]:
def split_into_chunks(text, chunk_size=500, overlap=50):
    """
    Split text into overlapping chunks.
    - chunk_size: number of characters per chunk
    - overlap: number of characters shared between consecutive chunks
    """
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

# Apply chunking to all documents
all_chunks = []       # list of chunk texts
chunk_metadata = []   # list of {source, chunk_index} for each chunk

for filename, text in documents.items():
    chunks = split_into_chunks(text, chunk_size=500, overlap=50)
    for i, chunk in enumerate(chunks):
        all_chunks.append(chunk)
        chunk_metadata.append({'source': filename, 'chunk_index': i})
    print(f'  🔪 {filename}: {len(chunks)} chunks')

print(f'\n✅ Total chunks created: {len(all_chunks)}')

## Step 5 — Preview a Chunk (Before Embedding)

In [ ]:
print('=' * 60)
print('📋 SAMPLE CHUNK (what gets fed into the embedding model):')
print('=' * 60)
print(f'Source: {chunk_metadata[0]["source"]}')
print(f'Chunk index: {chunk_metadata[0]["chunk_index"]}')
print(f'Character length: {len(all_chunks[0])}')
print()
print(all_chunks[0])

## Step 6 — Load the Embedding Model
We use `all-MiniLM-L6-v2` from HuggingFace.

### Why this model?
- ✅ **Free** — no API key or cost needed
- ✅ **Lightweight** — only 80MB, runs fast even on Colab CPU
- ✅ **High quality** — produces 384-dimension embeddings optimized for semantic similarity search
- ✅ **Widely used** — standard choice for RAG pipelines in production

In [ ]:
from sentence_transformers import SentenceTransformer

print('⏳ Loading embedding model: all-MiniLM-L6-v2...')
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print('✅ Model loaded!')
print(f'📐 Embedding dimensions: {model.get_sentence_embedding_dimension()}')

## Step 7 — Generate Embeddings
Convert all text chunks into embedding vectors.
Each chunk becomes a list of 384 numbers that capture its meaning.

In [ ]:
import numpy as np

print(f'⏳ Generating embeddings for {len(all_chunks)} chunks...')
print('(This may take 1–2 minutes on CPU)\n')

embeddings = model.encode(
    all_chunks,
    show_progress_bar=True,
    batch_size=32
)

print(f'\n✅ Embeddings generated!')
print(f'📊 Shape: {embeddings.shape}')  # (num_chunks, 384)
print(f'   → {embeddings.shape[0]} chunks × {embeddings.shape[1]} dimensions each')

## Step 8 — What Does an Embedding Look Like?
This shows the actual embedding vector of the first chunk.

In [ ]:
print('=' * 60)
print('🔢 SAMPLE EMBEDDING VECTOR (first 20 of 384 dimensions):')
print('=' * 60)
print(f'\nSource text (first 100 chars):')
print(f'  "{all_chunks[0][:100]}..."')
print(f'\nEmbedding vector (first 20 values):')
print(f'  {embeddings[0][:20].tolist()}')
print(f'\nFull vector shape: {embeddings[0].shape}')
print(f'Value range: min={embeddings[0].min():.4f}, max={embeddings[0].max():.4f}')
print()
print('💡 These numbers represent the MEANING of the text in 384-dimensional space.')
print('   Similar texts will have vectors that point in similar directions.')

## Step 9 — Semantic Similarity Demo
We test whether similar syllabus content produces similar embeddings by computing cosine similarity.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Test sentences
test_queries = [
    'What is the grading system for this course?',
    'When is the midterm exam?',
    'What are the attendance policies?',
]

print('🔍 SEMANTIC SIMILARITY DEMO')
print('Finding the most relevant chunk for each query...\n')

for query in test_queries:
    # Embed the query
    query_embedding = model.encode([query])

    # Compute cosine similarity against all chunks
    similarities = cosine_similarity(query_embedding, embeddings)[0]

    # Get top result
    top_idx = similarities.argmax()
    top_score = similarities[top_idx]
    top_source = chunk_metadata[top_idx]['source']

    print(f'Query : "{query}"')
    print(f'Source: {top_source}')
    print(f'Score : {top_score:.4f}')
    print(f'Chunk : "{all_chunks[top_idx][:200]}..."')
    print('-' * 60)

## Step 10 — Save Embeddings and Metadata
Save everything so it can be loaded in Checkpoint 2 (Vector Database Setup).

In [ ]:
import json

os.makedirs('embeddings_output', exist_ok=True)

# Save embeddings as numpy array
np.save('embeddings_output/syllabot_embeddings.npy', embeddings)

# Save chunks and metadata as JSON
output_data = [
    {
        'chunk_id': i,
        'source': chunk_metadata[i]['source'],
        'chunk_index': chunk_metadata[i]['chunk_index'],
        'text': all_chunks[i]
    }
    for i in range(len(all_chunks))
]

with open('embeddings_output/syllabot_chunks.json', 'w', encoding='utf-8') as f:
    json.dump(output_data, f, indent=2, ensure_ascii=False)

print('✅ Files saved to /embeddings_output/')
print(f'   syllabot_embeddings.npy — {embeddings.shape[0]} vectors × {embeddings.shape[1]} dims')
print(f'   syllabot_chunks.json   — {len(output_data)} chunk records')
print('\n📌 Keep these files — they are the input for Checkpoint 2 (Vector Database).')

## Step 11 — Download Output Files

In [ ]:
from google.colab import files

print('⬇️  Downloading embedding output files...')
files.download('embeddings_output/syllabot_embeddings.npy')
files.download('embeddings_output/syllabot_chunks.json')
print('✅ Download started!')

## Step 12 — Summary

In [ ]:
print('=' * 60)
print('📋 CHECKPOINT 1 — EMBEDDING GENERATION SUMMARY')
print('=' * 60)
print(f'Embedding model : sentence-transformers/all-MiniLM-L6-v2')
print(f'Model size      : ~80MB (lightweight, CPU-friendly)')
print(f'Dimensions      : 384 per chunk')
print(f'Chunk size      : 500 characters with 50-character overlap')
print(f'Documents       : {len(documents)}')
print(f'Total chunks    : {len(all_chunks)}')
print(f'Total embeddings: {embeddings.shape[0]} vectors')
print()
print('Why all-MiniLM-L6-v2?')
print('  - Free, no API key required')
print('  - Fast on CPU, ideal for Colab')
print('  - 384-dim vectors: compact but semantically rich')
print('  - Trained specifically for semantic similarity tasks')
print('  - Standard choice for RAG pipelines in production')
print()
print('Next step → Checkpoint 2: Load these embeddings into')
print('a vector database (ChromaDB or FAISS) for similarity search.')